# Chart Understanding Pipeline — Build From Scratch

**VelTech FDP Workshop**

Implement each stage of the pipeline using the pre-trained models and libraries provided.

| Step | Task |
|------|------|
| 1 | Install dependencies |
| 2 | Download weights + sample PDFs |
| 3 | Rasterize PDF pages to images |
| 4 | Detect chart regions (YOLO) |
| 5 | Classify chart type (YOLO-cls) |
| 6 | OCR each crop (PaddleOCR) |
| 7 | Assign semantic roles (Qwen2.5-VL) |
| 8 | Wire full pipeline |


## STEP 1 — Install dependencies

In [ ]:
# ==== PADDLEOCR CPU INSTALL : SAFE COLAB BLOCK ====

!python -V
!pip uninstall -y paddlepaddle paddlepaddle-gpu paddleocr paddlex || true
!pip install -U pip setuptools wheel

# Official Paddle CPU wheel index shown in PaddleOCR quick start
!python -m pip install paddlepaddle==3.2.0 -i https://www.paddlepaddle.org.cn/packages/stable/cpu/

# Install PaddleOCR after PaddlePaddle
!python -m pip install paddleocr

#RESTART THE SESSION

In [ ]:
# After Restarting the Session Run From Here
#==== CLEAN CONFLICTS ====

!pip uninstall -y \
  langchain \
  langchain-core \
  langchain-community \
  langchain-text-splitters \
  langsmith \
  paddleocr \
  paddlex \
  paddlepaddle \
  paddlepaddle-gpu

In [ ]:
# ==== INSTALL PADDLE OCR (CPU SAFE) ====

!python -V
!pip install -U pip setuptools wheel

# Official Paddle CPU install
!python -m pip install paddlepaddle==3.2.0 -i https://www.paddlepaddle.org.cn/packages/stable/cpu/

# Then PaddleOCR
!python -m pip install paddleocr==3.4.0

In [ ]:
# ==== Qwen2.5-VL ====
!pip install -q transformers accelerate qwen-vl-utils

# ==== YOLO + PDF + Drive download ====
!pip install -q ultralytics opencv-python pillow matplotlib pandas tqdm
!pip install -q pymupdf gdown

print("✅ Other dependencies installed")
!nvidia-smi | head -10

## STEP 2 — Download weights and sample PDFs

All model weights and sample PDFs are available via the links below.

In [ ]:
# ── Config: all external links in one place ───────────────────────────────
DRIVE_IDS = {
    "demo_bundle":        "YOUR_BUNDLE_DRIVE_ID",   # weights + sample PDFs zip
    "sample_papers":     "1m4MgUIP93ds8Jr7lMyp6csSKPPneew5w",
}

WEIGHT_FILES = {
    "chart_detector":    "chart_detector_v3.pt",
    "plot_classifier":   "Plot_Classifier_new_latest.pt",
    "bar_detection":     "Bar_Detection_Yolo_v4.pt",
    "line_plot_area":    "plot_area_detector_line_v1.pt",
    "line_embed":        "best_line_embed_instance_v1.pt",
}

# Tunable parameters
PAGE_ZOOM            = 2.0
DETECT_CONF          = 0.25
DETECT_IOU           = 0.45
CROP_PAD             = 12
CLASSIFIER_MIN_CONF  = 0.50
OCR_MIN_CONF         = 0.60
MAX_OCR_BOXES        = 18


In [ ]:
from pathlib import Path
import zipfile, shutil

ROOT = Path("/content/chart_demo")
ZIP_PATH = Path("/content/chart_bar_line_qwen_demo_bundle.zip")

# Demo bundle Google Drive ID (replace with your own if you re-package)
DRIVE_FILE_ID = "185upEGS0nI_IjGxoneglgjIfKBGvRNnU"

if ROOT.exists():
    shutil.rmtree(ROOT)
ROOT.mkdir(parents=True, exist_ok=True)

!gdown --id "$DRIVE_FILE_ID" -O "$ZIP_PATH"

with zipfile.ZipFile(ZIP_PATH, "r") as z:
    z.extractall(ROOT)

# Normalize any Windows-style backslash paths inside the zip
for p in list(ROOT.rglob("*")):
    if p.is_file() and "\\" in p.name:
        parts = p.name.split("\\")
        new_path = p.parent.joinpath(*parts)
        new_path.parent.mkdir(parents=True, exist_ok=True)
        if not new_path.exists():
            shutil.move(str(p), str(new_path))

print("✅ Extracted to:", ROOT)
for p in sorted(ROOT.iterdir())[:15]:
    print(" -", p.name)

## STEP 3 — Rasterize PDF pages to images

In [ ]:
# Open the PDF and convert every page to a PNG image.
# Save to /content/chart_demo_outputs/pdf_pages/
# Libraries: fitz (PyMuPDF), pathlib


## STEP 4 — Detect chart regions and crop

In [ ]:
# For each page image, run the chart detector (YOLO).
# Crop each detected region (add CROP_PAD padding).
# Save crops to /content/chart_demo_outputs/chart_crops/
# Save metadata to chart_crops_metadata.csv


## STEP 5 — Classify chart type

In [ ]:
# For each crop, run the plot classifier (YOLO-cls).
# Record pred_chart_type and pred_chart_type_confidence.
# Save to chart_crops_classified_metadata.csv


## STEP 6 — OCR each crop

In [ ]:
# Run PaddleOCR on each crop.
# For each detected text region, extract:
#   text, bounding box, confidence, angle
# Featurize each box: position band, numeric/year/rotated flags.


## STEP 7 — Assign semantic roles with Qwen2.5-VL

In [ ]:
# Load Qwen2.5-VL. For each crop, send the OCR boxes + image to Qwen.
# Ask it to assign a role to each box:
#   title, x_tick, y_tick, data_label, legend, caption, other
# Parse the JSON response and attach roles to boxes.


## STEP 8 — Wire the full pipeline

In [ ]:
# For each bar/line crop:
#   A. Run OCR + featurize
#   B. Assign roles via Qwen
#   C. Draw coloured bounding boxes and save overlay
#   D. Run the YOLO extractor (bar_det or line_plot_area)
# Display results in a grid.
